# **(ADD HERE THE NOTEBOOK NAME)**

## Objectives

* Load the trained model
* Evaluate performance
* Calculate classification accuracy and loss
* Analyse predictions using a confusion matrixx and classification report 
* Determine wherther the model meets 97% accuracy rate

## Inputs

* Trained model
* Test images

## Outputs

* Test accuracy and loss 
* Confusion matrix 
* Precision recall and F1 score
* Evaluation findings for the Streamlit dashboard

## Additional Comments

* The test dataset has not been used during model training or model selection. It provides an independent assessment of model performance.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [3]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/FInal-Project-01/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'/workspaces/FInal-Project-01'

# Section 1

Section 1 content

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

print("TensorFlow version:", tf.__version__)
print("Current directory:", Path.cwd())

TensorFlow version: 2.16.1
Current directory: /workspaces/FInal-Project-01


In [7]:
model_path = Path("outputs/models/cherry_leaf_cnn.keras")

model = tf.keras.models.load_model(model_path)

print("Model loaded successfully!")
print("Model input shape:", model.input_shape)

Model loaded successfully!
Model input shape: (None, 100, 100, 3)


In [8]:
test_path = Path("data/processed/test")

IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    labels="inferred",
    label_mode="binary",
    class_names=["healthy", "powdery_mildew"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False 
)

print("Class names:", test_dataset.class_names)

Found 646 files belonging to 2 classes.
Class names: ['healthy', 'powdery_mildew']


In [9]:
test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Test Loss: {test_loss:.4f}")

target_accuracy = 0.97

if test_accuracy >= target_accuracy:
    print("The model meets the 97% accuracy target.")
else:
    print("The model does not yet meet the 97% accuracy target.")

21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.9969 - loss: 0.0289
Test Accuracy: 99.69%
Test Loss: 0.0289
The model meets the 97% accuracy target.


---

# Confusion Matrix

A confusion matrix is used to examine the model's predictions
for each class.

This helps identify whether the model is more likely to
misclassify healthy leaves or leaves affected by powdery mildew.

Section 2 content

In [10]:
# Collect the actual labels
y_true = np.concatenate([
    labels.numpy().flatten()
    for images, labels in test_dataset
]).astype(int)

# Generate predicted probabilities
y_probabilities = model.predict(test_dataset).flatten()

# Convert probabilities into class predictions
y_pred = (y_probabilities >= 0.5).astype(int)

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Healthy", "Powdery Mildew"],
    yticklabels=["Healthy", "Powdery Mildew"]
)

plt.title("Confusion Matrix — Cherry Leaf CNN")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.show()

 1/21 ━━━━━━━━━━━━━━━━━━━━ 3s 168ms/step

2026-10-12 14:30:18.574651: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step


<Figure size 700x500 with 2 Axes>

In [11]:
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Healthy", "Powdery Mildew"],
        digits=4
    )
)

Confusion Matrix:
[[330   0]
 [  2 314]]

Classification Report:
                precision    recall  f1-score   support

       Healthy     0.9940    1.0000    0.9970       330
Powdery Mildew     1.0000    0.9937    0.9968       316

      accuracy                         0.9969       646
     macro avg     0.9970    0.9968    0.9969       646
  weighted avg     0.9969    0.9969    0.9969       646



In [12]:
from pathlib import Path
import json

evaluation_path = Path("outputs/evaluation")
evaluation_path.mkdir(parents=True, exist_ok=True)

# Save overall model performance
metrics = {
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "target_accuracy": 0.97,
    "target_met": bool(test_accuracy >= 0.97),
    "test_images": int(len(y_true))
}

with open(evaluation_path / "test_metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

# Save the classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=["Healthy", "Powdery Mildew"],
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    evaluation_path / "classification_report.csv"
)

# Save the confusion matrix
pd.DataFrame(
    cm,
    index=["Actual Healthy", "Actual Powdery Mildew"],
    columns=["Predicted Healthy", "Predicted Powdery Mildew"]
).to_csv(evaluation_path / "confusion_matrix.csv")

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


## Model Evaluation Findings

The CNN achieved a test accuracy of **99.69%**, exceeding
the client's minimum requirement of 97%.

### Key Results
- Test images evaluated: 646
- Correct classifications: 644
- Incorrect classifications: 2
- Test accuracy: 99.69%
- Test loss: 0.0289
- Healthy recall: 100%
- Powdery mildew recall: 99.37%

### Interpretation
The model successfully distinguished healthy leaves from
powdery mildew in nearly all test images.

Both incorrect predictions were powdery mildew leaves
classified as healthy (false negatives).

This is an important limitation because failing to identify
a diseased leaf could delay appropriate action.

### Business Requirement
The model exceeds the required 97% classification accuracy
on the held-out test dataset.

### Limitations
The dataset contains augmented images. Source-image grouping
was used during splitting to reduce data leakage.

Further checks for duplicate images and evaluation on new,
independently collected photographs would strengthen
confidence in the model's generalisation

## Data Leakage Check

The dataset was split using source-image groups to keep known
augmented versions of each photograph together.

An additional SHA-256 hash check is performed to identify
identical image files appearing across training, validation,
and test datasets.

This check detects exact file duplicates but does not detect
visually identical images saved with different compression,
rotations, or other transformations.

In [13]:
import hashlib
from collections import defaultdict
from pathlib import Path

processed_path = Path("data/processed")

split_names = ["train", "validation", "test"]
supported_extensions = {".jpg", ".jpeg", ".png", ".webp"}

# Store which dataset splits contain each image hash
hash_locations = defaultdict(set)

for split_name in split_names:
    split_folder = processed_path / split_name

    for image_path in split_folder.rglob("*"):
        if (
            image_path.is_file()
            and image_path.suffix.lower() in supported_extensions
        ):
            image_hash = hashlib.sha256(
                image_path.read_bytes()
            ).hexdigest()

            hash_locations[image_hash].add(split_name)

# Identify identical files appearing in different splits
cross_split_duplicates = {
    image_hash: locations
    for image_hash, locations in hash_locations.items()
    if len(locations) > 1
}

print("Unique file hashes:", len(hash_locations))
print(
    "Exact duplicate hashes across splits:",
    len(cross_split_duplicates)
)

Unique file hashes: 4208
Exact duplicate hashes across splits: 0


---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* If you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
    # create here your folder
    # os.makedirs(name='')
except Exception as e:
    print(e)
